In [1]:
import sys
!{sys.executable} -m pip install pandas numpy


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [3]:
import os
os.chdir('..')
print(os.getcwd())

C:\Users\91935\PycharmProjects\AQi project


In [4]:
city_day = pd.read_csv('data/raw/kaggle/city_day.csv')
station_day = pd.read_csv('data/raw/kaggle/station_day.csv')  
openaq = pd.read_csv('data/raw/openaq/openaq_recent.csv')
weather_hist = pd.read_csv('data/raw/weather/weather_2015_2020.csv')   
weather_recent = pd.read_csv('data/raw/weather/weather_recent.csv')  

In [5]:
for name, df in [('city_day', city_day), ('openaq', openaq), ('weather_hist', weather_hist), ('weather_recent', weather_recent)]:
    print(name, df.shape, df.columns.tolist())

city_day (29531, 16) ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']
openaq (96350, 8) ['sensor_id', 'location_id', 'location_name', 'city', 'parameter', 'date', 'value', 'coverage_pct']
weather_hist (14063, 7) ['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'city']
weather_recent (2625, 7) ['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'city']


In [6]:
city_mapping = {
    'Bengaluru': 'Bengaluru', 'Bangalore': 'Bengaluru',
    'Delhi': 'Delhi', 'New Delhi': 'Delhi',
    'Mumbai': 'Mumbai', 'Bombay': 'Mumbai',
    'Chennai': 'Chennai', 'Madras': 'Chennai',
    'Kolkata': 'Kolkata', 'Calcutta': 'Kolkata',
    'Lucknow': 'Lucknow',
    'Patna': 'Patna',
}

def standardize_city(df, col):
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].map(city_mapping).fillna(df[col])
    return df

city_day = standardize_city(city_day, 'City')
openaq = standardize_city(openaq, 'city')
weather_hist = standardize_city(weather_hist, 'city')
weather_recent = standardize_city(weather_recent, 'city')

target_cities = {'Lucknow', 'Mumbai', 'Patna', 'Bengaluru', 'Delhi', 'Chennai', 'Kolkata'}
weather_hist = weather_hist[weather_hist['city'].isin(target_cities)].copy()

print(sorted(city_day['City'].unique()))
print(sorted(openaq['city'].unique()))
print(sorted(weather_hist['city'].unique()))
print(sorted(weather_recent['city'].unique()))

['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru', 'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Ernakulam', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna', 'Shillong', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']
['Bengaluru', 'Chennai', 'Delhi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna']
['Bengaluru', 'Chennai', 'Delhi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna']
['Bengaluru', 'Chennai', 'Delhi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna']


In [7]:
print(openaq['parameter'].unique())

['o3' 'pm10' 'pm25' 'so2' 'co' 'no2']


In [8]:
target_cities = {'Lucknow', 'Mumbai', 'Patna', 'Bengaluru', 'Delhi', 'Chennai', 'Kolkata'}

weather_hist = weather_hist[weather_hist['city'].isin(target_cities)].copy()
print(weather_hist['city'].unique())  

['Delhi' 'Lucknow' 'Patna' 'Mumbai' 'Bengaluru' 'Chennai' 'Kolkata']


In [9]:
weather_recent = weather_recent[weather_recent['city'].isin(target_cities)].copy()
print(weather_recent['city'].unique())  

['Delhi' 'Lucknow' 'Patna' 'Mumbai' 'Bengaluru' 'Chennai' 'Kolkata']


In [10]:
print(set(weather_recent['city'].unique()))

{'Bengaluru', 'Patna', 'Kolkata', 'Chennai', 'Mumbai', 'Delhi', 'Lucknow'}


In [11]:
param_mapping = {
    'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2',
    'so2': 'SO2', 'co': 'CO', 'o3': 'O3',
}
openaq['parameter'] = openaq['parameter'].map(param_mapping).fillna(openaq['parameter'])

openaq_wide = openaq.pivot_table(
    index=['city', 'date'],
    columns='parameter',
    values='value',
    aggfunc='mean'
).reset_index()

print(openaq_wide.shape)
print(openaq_wide.columns.tolist())

(2566, 8)
['city', 'date', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2']


In [12]:
def add_date_features(df, date_col):
    df[date_col] = pd.to_datetime(df[date_col])
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day_of_week'] = df[date_col].dt.dayofweek
    df['season'] = df['month'].map(lambda m:
        'winter' if m in [11, 12, 1, 2] else
        'summer' if m in [3, 4, 5, 6] else
        'monsoon' if m in [7, 8, 9] else
        'post_monsoon'
    )
    return df

city_day = add_date_features(city_day, 'Date')
openaq_wide = add_date_features(openaq_wide, 'date')
weather_hist = add_date_features(weather_hist, 'date')
weather_recent = add_date_features(weather_recent, 'date')

print(city_day['Date'].dtype)
print(city_day[['Date','year','month','day_of_week','season']].head())

datetime64[ns]
        Date  year  month  day_of_week  season
0 2015-01-01  2015      1            3  winter
1 2015-01-02  2015      1            4  winter
2 2015-01-03  2015      1            5  winter
3 2015-01-04  2015      1            6  winter
4 2015-01-05  2015      1            0  winter


In [13]:
for name, df, col in [('city_day', city_day, 'Date'), ('openaq_wide', openaq_wide, 'date'),
                       ('weather_hist', weather_hist, 'date'), ('weather_recent', weather_recent, 'date')]:
    print(name, df[col].min(), '→', df[col].max())

city_day 2015-01-01 00:00:00 → 2020-07-01 00:00:00
openaq_wide 2025-08-01 00:00:00 → 2026-08-10 00:00:00
weather_hist 2015-01-01 00:00:00 → 2020-07-01 00:00:00
weather_recent 2025-08-01 00:00:00 → 2026-08-10 00:00:00


In [14]:
pollutant_cols_kaggle = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
pollutant_cols_openaq = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']

def apply_missing_data_rule(df, group_col, date_col, pollutant_cols, drop_threshold=40, ffill_limit=2):
    df = df.sort_values([group_col, date_col])
    dropped_log = []
    for city in df[group_col].unique():
        mask = df[group_col] == city
        for col in pollutant_cols:
            pct_missing = df.loc[mask, col].isna().mean() * 100
            if pct_missing >= drop_threshold:
                dropped_log.append((city, col, round(pct_missing, 1)))
                df.loc[mask, col] = np.nan
    df[pollutant_cols] = df.groupby(group_col)[pollutant_cols].transform(
        lambda x: x.ffill(limit=ffill_limit)
    )
    return df, dropped_log

city_day, dropped_kaggle = apply_missing_data_rule(city_day, 'City', 'Date', pollutant_cols_kaggle)
openaq_wide, dropped_openaq = apply_missing_data_rule(openaq_wide, 'city', 'date', pollutant_cols_openaq)

print("Dropped (Kaggle):", dropped_kaggle)
print("Dropped (OpenAQ):", dropped_openaq)

Dropped (Kaggle): [('Ahmedabad', 'PM10', np.float64(79.7)), ('Ahmedabad', 'NH3', np.float64(100.0)), ('Aizawl', 'Xylene', np.float64(100.0)), ('Bengaluru', 'Xylene', np.float64(100.0)), ('Bhopal', 'Benzene', np.float64(100.0)), ('Bhopal', 'Toluene', np.float64(100.0)), ('Bhopal', 'Xylene', np.float64(100.0)), ('Brajrajnagar', 'Toluene', np.float64(100.0)), ('Brajrajnagar', 'Xylene', np.float64(100.0)), ('Chennai', 'PM10', np.float64(85.0)), ('Chennai', 'Xylene', np.float64(100.0)), ('Coimbatore', 'Xylene', np.float64(100.0)), ('Ernakulam', 'O3', np.float64(100.0)), ('Gurugram', 'PM10', np.float64(55.9)), ('Gurugram', 'NH3', np.float64(92.9)), ('Gurugram', 'Xylene', np.float64(92.9)), ('Guwahati', 'Toluene', np.float64(100.0)), ('Guwahati', 'Xylene', np.float64(100.0)), ('Jaipur', 'Xylene', np.float64(100.0)), ('Jorapokhar', 'PM2.5', np.float64(67.6)), ('Jorapokhar', 'NOx', np.float64(100.0)), ('Jorapokhar', 'CO', np.float64(43.9)), ('Jorapokhar', 'Benzene', np.float64(100.0)), ('Jorapo

In [15]:
mumbai = city_day[city_day['City']=='Mumbai'][['Date','PM2.5']].copy()
mumbai['is_missing'] = mumbai['PM2.5'].isna()

monthly_missing = mumbai.set_index('Date')['is_missing'].resample('ME').mean()
print(monthly_missing)

Date
2015-01-31    1.0
2015-02-28    1.0
2015-03-31    1.0
2015-04-30    1.0
2015-05-31    1.0
             ... 
2020-03-31    1.0
2020-04-30    1.0
2020-05-31    1.0
2020-06-30    1.0
2020-07-31    1.0
Freq: ME, Name: is_missing, Length: 67, dtype: float64


In [16]:
city_day_raw = pd.read_csv('data/raw/kaggle/city_day.csv')
city_day_raw = standardize_city(city_day_raw, 'City')
city_day_raw['Date'] = pd.to_datetime(city_day_raw['Date'])

mumbai_raw = city_day_raw[city_day_raw['City']=='Mumbai'][['Date','PM2.5']].copy()
mumbai_raw['is_missing'] = mumbai_raw['PM2.5'].isna()

monthly_missing_raw = mumbai_raw.set_index('Date')['is_missing'].resample('ME').mean()
print(monthly_missing_raw.value_counts())
print()
print("Months with SOME data (missing < 1.0):")
print(monthly_missing_raw[monthly_missing_raw < 1.0])

is_missing
1.000000    39
0.000000    23
0.966667     1
0.161290     1
0.032258     1
0.064516     1
0.066667     1
Name: count, dtype: int64

Months with SOME data (missing < 1.0):
Date
2018-04-30    0.966667
2018-05-31    0.161290
2018-06-30    0.000000
2018-07-31    0.032258
2018-08-31    0.000000
2018-09-30    0.000000
2018-10-31    0.064516
2018-11-30    0.066667
2018-12-31    0.000000
2019-01-31    0.000000
2019-02-28    0.000000
2019-03-31    0.000000
2019-04-30    0.000000
2019-05-31    0.000000
2019-06-30    0.000000
2019-07-31    0.000000
2019-08-31    0.000000
2019-09-30    0.000000
2019-10-31    0.000000
2019-11-30    0.000000
2019-12-31    0.000000
2020-01-31    0.000000
2020-02-29    0.000000
2020-03-31    0.000000
2020-04-30    0.000000
2020-05-31    0.000000
2020-06-30    0.000000
2020-07-31    0.000000
Freq: ME, Name: is_missing, dtype: float64


In [17]:
# find the first date each pollutant actually has real data, per city
def first_valid_date_per_city(df, group_col, date_col, cols):
    result = {}
    for city in df[group_col].unique():
        sub = df[df[group_col] == city]
        result[city] = {col: sub.loc[sub[col].notna(), date_col].min() for col in cols}
    return result

mumbai_start_dates = first_valid_date_per_city(city_day_raw, 'City', 'Date', ['PM2.5','PM10','NO','NO2','SO2','O3'])
print(mumbai_start_dates['Mumbai'])

{'PM2.5': Timestamp('2018-04-26 00:00:00'), 'PM10': Timestamp('2018-04-26 00:00:00'), 'NO': Timestamp('2018-01-16 00:00:00'), 'NO2': Timestamp('2018-02-27 00:00:00'), 'SO2': Timestamp('2017-12-08 00:00:00'), 'O3': Timestamp('2017-12-29 00:00:00')}


In [18]:
def apply_missing_data_rule_v2(df, group_col, date_col, pollutant_cols, drop_threshold=40, ffill_limit=2):
    df = df.sort_values([group_col, date_col]).copy()
    dropped_log = []
    trimmed_log = []

    for city in df[group_col].unique():
        mask = df[group_col] == city
        for col in pollutant_cols:
            city_data = df.loc[mask, [date_col, col]]
            first_valid = city_data.loc[city_data[col].notna(), date_col].min()

            if pd.isna(first_valid):
                # never measured at all for this city — drop entirely
                dropped_log.append((city, col, 100.0))
                continue

            # only evaluate missing % from first_valid onward
            active_window = city_data[city_data[date_col] >= first_valid]
            pct_missing = active_window[col].isna().mean() * 100

            if pct_missing >= drop_threshold:
                dropped_log.append((city, col, round(pct_missing, 1)))
                df.loc[mask, col] = np.nan
            elif first_valid > df.loc[mask, date_col].min():
                trimmed_log.append((city, col, first_valid.date()))

    df[pollutant_cols] = df.groupby(group_col)[pollutant_cols].transform(
        lambda x: x.ffill(limit=ffill_limit)
    )
    return df, dropped_log, trimmed_log

city_day, dropped_kaggle_v2, trimmed_kaggle = apply_missing_data_rule_v2(city_day_raw, 'City', 'Date', pollutant_cols_kaggle)

print("Dropped (still ≥40% missing within active window):")
print(dropped_kaggle_v2)
print()
print("Trimmed (pollutant started mid-series, not counted as missing beforehand):")
print(trimmed_kaggle)

Dropped (still ≥40% missing within active window):
[('Ahmedabad', 'NH3', 100.0), ('Aizawl', 'Xylene', 100.0), ('Bengaluru', 'Xylene', 100.0), ('Bhopal', 'Benzene', 100.0), ('Bhopal', 'Toluene', 100.0), ('Bhopal', 'Xylene', 100.0), ('Brajrajnagar', 'Toluene', 100.0), ('Brajrajnagar', 'Xylene', 100.0), ('Chennai', 'Xylene', 100.0), ('Coimbatore', 'Xylene', 100.0), ('Ernakulam', 'O3', 100.0), ('Guwahati', 'Toluene', 100.0), ('Guwahati', 'Xylene', 100.0), ('Jaipur', 'Xylene', 100.0), ('Jorapokhar', 'NOx', 100.0), ('Jorapokhar', 'CO', np.float64(43.9)), ('Jorapokhar', 'Benzene', 100.0), ('Jorapokhar', 'Toluene', 100.0), ('Jorapokhar', 'Xylene', 100.0), ('Kochi', 'Benzene', 100.0), ('Kochi', 'Xylene', 100.0), ('Lucknow', 'PM10', 100.0), ('Lucknow', 'Xylene', 100.0), ('Mumbai', 'Toluene', np.float64(49.4)), ('Mumbai', 'Xylene', np.float64(49.5)), ('Shillong', 'Xylene', 100.0), ('Talcher', 'Toluene', np.float64(80.8)), ('Talcher', 'Xylene', 100.0), ('Thiruvananthapuram', 'Benzene', 100.0), ('T

In [19]:
openaq_wide, dropped_openaq_v2, trimmed_openaq = apply_missing_data_rule_v2(
    openaq_wide, 'city', 'date', pollutant_cols_openaq
)

print("Dropped (OpenAQ):", dropped_openaq_v2)
print()
print("Trimmed (OpenAQ):", trimmed_openaq)

Dropped (OpenAQ): [('Patna', 'NO2', 100.0), ('Patna', 'SO2', 100.0), ('Patna', 'CO', 100.0)]

Trimmed (OpenAQ): []


In [20]:
merged_historical = pd.merge(
    city_day, weather_hist,
    left_on=['City', 'Date'], right_on=['city', 'date'],
    how='inner', suffixes=('', '_weather')
)
merged_historical['data_source'] = 'kaggle_historical'

print(f"city_day rows: {len(city_day)}")
print(f"weather_hist rows: {len(weather_hist)}")
print(f"merged_historical rows: {len(merged_historical)}")
print(f"Rows dropped from city_day: {len(city_day) - len(merged_historical)}")

city_day rows: 29531
weather_hist rows: 14063
merged_historical rows: 12717
Rows dropped from city_day: 16814


In [21]:
city_day_target = city_day[city_day['City'].isin(target_cities)]
print(f"city_day rows for target cities only: {len(city_day_target)}")
print(f"weather_hist rows: {len(weather_hist)}")

# checking just the 7 target cities, how many actually matched?
merged_target_only = pd.merge(
    city_day_target, weather_hist, 
    left_on=['City', 'Date'], right_on=['city', 'date'],
    how='inner'
)
print(f"merged rows (target cities only): {len(merged_target_only)}")
print(f"Rows dropped even within target cities: {len(city_day_target) - len(merged_target_only)}")

city_day rows for target cities only: 12717
weather_hist rows: 14063
merged rows (target cities only): 12717
Rows dropped even within target cities: 0


In [22]:
city_day = city_day[city_day['City'].isin(target_cities)].copy()

merged_historical = pd.merge(
    city_day, weather_hist,
    left_on=['City', 'Date'], right_on=['city', 'date'],
    how='inner', suffixes=('', '_weather')
)
merged_historical['data_source'] = 'kaggle_historical'

print(f"city_day rows (target cities only): {len(city_day)}")
print(f"weather_hist rows: {len(weather_hist)}")
print(f"merged_historical rows: {len(merged_historical)}")
print(f"Rows dropped: {len(city_day) - len(merged_historical)}")

city_day rows (target cities only): 12717
weather_hist rows: 14063
merged_historical rows: 12717
Rows dropped: 0


In [23]:
merged_recent = pd.merge(
    openaq_wide, weather_recent,
    left_on=['city', 'date'], right_on=['city', 'date'],
    how='inner', suffixes=('', '_weather')
)
merged_recent['data_source'] = 'openaq_recent'

print(f"openaq_wide rows: {len(openaq_wide)}")
print(f"weather_recent rows: {len(weather_recent)}")
print(f"merged_recent rows: {len(merged_recent)}")
print(f"Rows dropped: {len(openaq_wide) - len(merged_recent)}")

openaq_wide rows: 2566
weather_recent rows: 2625
merged_recent rows: 2566
Rows dropped: 0


In [24]:
merged_historical = merged_historical.rename(columns={'City': 'city', 'Date': 'date'})

print("merged_historical columns:", merged_historical.columns.tolist())
print()
print("merged_recent columns:", merged_recent.columns.tolist())

merged_historical columns: ['city', 'date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'city', 'year', 'month', 'day_of_week', 'season', 'data_source']

merged_recent columns: ['city', 'date', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'year', 'month', 'day_of_week', 'season', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'year_weather', 'month_weather', 'day_of_week_weather', 'season_weather', 'data_source']


In [25]:
weather_hist_renamed = weather_hist.rename(columns={'city': 'City', 'date': 'Date'})

merged_historical = pd.merge(
    city_day, weather_hist_renamed,
    on=['City', 'Date'],
    how='inner'
)
merged_historical['data_source'] = 'kaggle_historical'
merged_historical = merged_historical.rename(columns={'City': 'city', 'Date': 'date'})

print(merged_historical.columns.tolist())
print(len(merged_historical))

['city', 'date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'year', 'month', 'day_of_week', 'season', 'data_source']
12717


In [26]:
# drop date-feature columns from weather_recent before merging — openaq_wide's versions are the ones we'll keep
weather_recent_trimmed = weather_recent.drop(columns=['year', 'month', 'day_of_week', 'season'])

merged_recent = pd.merge(
    openaq_wide, weather_recent_trimmed,
    on=['city', 'date'],
    how='inner'
)
merged_recent['data_source'] = 'openaq_recent'

print(merged_recent.columns.tolist())
print(len(merged_recent))

['city', 'date', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'year', 'month', 'day_of_week', 'season', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'relative_humidity_2m_mean', 'data_source']
2566


In [27]:
for col in pollutant_cols_openaq:
    print(col, merged_recent[col].describe()[['min','mean','max']].to_dict())

PM2.5 {'min': 3.9266666666666663, 'mean': 81.04160607658326, 'max': 11705.95}
PM10 {'min': 7.359999999999999, 'mean': 151.63375934853383, 'max': 17668.47142857143}
NO2 {'min': -78.60363636363637, 'mean': 27.307552751077417, 'max': 177.0}
SO2 {'min': 0.92, 'mean': 8.636291976280052, 'max': 52.8}
CO {'min': -98.37408333333333, 'mean': 0.8115761865047778, 'max': 7.563571428571429}
O3 {'min': -76.55, 'mean': 25.203942440757206, 'max': 94.8}


In [28]:
print("Negative value counts:")
for col in ['NO2', 'CO', 'O3']:
    neg_count = (merged_recent[col] < 0).sum()
    print(f"  {col}: {neg_count} rows ({neg_count/len(merged_recent)*100:.1f}%)")

print()
print("Extreme outlier counts (using generous real-world ceilings):")
ceilings = {'PM2.5': 1000, 'PM10': 1000, 'NO2': 500, 'SO2': 500, 'CO': 50, 'O3': 500}
for col, ceiling in ceilings.items():
    over_count = (merged_recent[col] > ceiling).sum()
    print(f"  {col} > {ceiling}: {over_count} rows ({over_count/len(merged_recent)*100:.1f}%)")

Negative value counts:
  NO2: 2 rows (0.1%)
  CO: 2 rows (0.1%)
  O3: 2 rows (0.1%)

Extreme outlier counts (using generous real-world ceilings):
  PM2.5 > 1000: 11 rows (0.4%)
  PM10 > 1000: 12 rows (0.5%)
  NO2 > 500: 0 rows (0.0%)
  SO2 > 500: 0 rows (0.0%)
  CO > 50: 0 rows (0.0%)
  O3 > 500: 0 rows (0.0%)


In [29]:
# define validity bounds per pollutant
validity_bounds = {
    'PM2.5': (0, 1000),
    'PM10': (0, 1000),
    'NO2': (0, 500),
    'SO2': (0, 500),
    'CO': (0, 50),
    'O3': (0, 500),
}

invalid_counts = {}
for col, (low, high) in validity_bounds.items():
    invalid_mask = (merged_recent[col] < low) | (merged_recent[col] > high)
    invalid_counts[col] = invalid_mask.sum()
    merged_recent.loc[invalid_mask, col] = np.nan

print("Rows set to NaN per column:", invalid_counts)

# re-run missing-data rule on merged_recent now that some values are freshly NaN
merged_recent, dropped_after_cleanup, trimmed_after_cleanup = apply_missing_data_rule_v2(
    merged_recent, 'city', 'date', pollutant_cols_openaq
)
print("Dropped after outlier cleanup:", dropped_after_cleanup)

Rows set to NaN per column: {'PM2.5': np.int64(11), 'PM10': np.int64(12), 'NO2': np.int64(2), 'SO2': np.int64(0), 'CO': np.int64(2), 'O3': np.int64(2)}
Dropped after outlier cleanup: [('Patna', 'NO2', 100.0), ('Patna', 'SO2', 100.0), ('Patna', 'CO', 100.0)]


In [30]:
CPCB_BREAKPOINTS = {
    'PM2.5': [(0,30,0,50),(31,60,51,100),(61,90,101,200),(91,120,201,300),(121,250,301,400),(251,380,401,500)],
    'PM10':  [(0,50,0,50),(51,100,51,100),(101,250,101,200),(251,350,201,300),(351,430,301,400),(431,600,401,500)],
    'NO2':   [(0,40,0,50),(41,80,51,100),(81,180,101,200),(181,280,201,300),(281,400,301,400),(401,500,401,500)],
    'SO2':   [(0,40,0,50),(41,80,51,100),(81,380,101,200),(381,800,201,300),(801,1600,301,400),(1601,2000,401,500)],
    'CO':    [(0,1.0,0,50),(1.1,2.0,51,100),(2.1,10,101,200),(10.1,17,201,300),(17.1,34,301,400),(34.1,50,401,500)],
    'O3':    [(0,50,0,50),(51,100,51,100),(101,168,101,200),(169,208,201,300),(209,748,301,400),(749,1000,401,500)],
}

def sub_index(pollutant, conc):
    if pd.isna(conc):
        return np.nan
    breakpoints = CPCB_BREAKPOINTS[pollutant]
    for c_lo, c_hi, i_lo, i_hi in breakpoints:
        if c_lo <= conc <= c_hi:
            return ((i_hi - i_lo) / (c_hi - c_lo)) * (conc - c_lo) + i_lo
    # concentration exceeds the highest defined breakpoint
    if conc > breakpoints[-1][1]:
        return 500.0
    return np.nan

def compute_aqi(row):
    sub_indices = []
    for pollutant in CPCB_BREAKPOINTS:
        if pollutant in row.index:
            si = sub_index(pollutant, row[pollutant])
            if not pd.isna(si):
                sub_indices.append(si)
    return max(sub_indices) if sub_indices else np.nan

# quick test on a few known reference points before applying to real data
print(sub_index('PM2.5', 15))   # should be in the 0-50 "Good" band
print(sub_index('PM2.5', 45))   # should be in the 51-100 "Satisfactory" band
print(sub_index('PM2.5', 9999)) # should cap at 500 (beyond all breakpoints)


25.0
74.65517241379311
500.0


In [31]:
merged_recent['AQI'] = merged_recent.apply(compute_aqi, axis=1)

print(merged_recent['AQI'].describe())
print()
print("Rows with no computable AQI (all pollutants NaN):", merged_recent['AQI'].isna().sum())

count    2566.000000
mean      114.217641
std        79.217715
min        22.922778
25%        60.139474
50%        92.333333
75%       135.327051
max       500.000000
Name: AQI, dtype: float64

Rows with no computable AQI (all pollutants NaN): 0


In [32]:
print(merged_recent.groupby('city')['AQI'].describe()[['count','mean','min','max']])

           count        mean        min         max
city                                               
Bengaluru  365.0   77.026376  28.950000  228.808759
Chennai    366.0   89.161926  26.216667  485.648837
Delhi      375.0  203.411676  37.485000  500.000000
Kolkata    365.0   95.289457  24.625000  322.277326
Lucknow    365.0  120.936278  34.400000  347.813953
Mumbai     365.0  105.031704  22.922778  500.000000
Patna      365.0  106.291043  32.000000  301.000000


In [33]:

common_cols = ['city', 'date', 'year', 'month', 'season', 'day_of_week',
               'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI', 'data_source',
               'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
               'windspeed_10m_max', 'relative_humidity_2m_mean']

master = pd.concat([merged_historical[common_cols], merged_recent[common_cols]], ignore_index=True)

print(master.shape)
print(master['data_source'].value_counts())
print()
print(master.head())
print()
print(master.tail())

(15283, 19)
data_source
kaggle_historical    12717
openaq_recent         2566
Name: count, dtype: int64

        city       date  year  month  season  day_of_week  PM2.5  PM10    NO2  \
0  Bengaluru 2015-01-01  2015      1  winter            3    NaN   NaN  17.33   
1  Bengaluru 2015-01-02  2015      1  winter            4    NaN   NaN  19.73   
2  Bengaluru 2015-01-03  2015      1  winter            5    NaN   NaN  19.88   
3  Bengaluru 2015-01-04  2015      1  winter            6    NaN   NaN  21.61   
4  Bengaluru 2015-01-05  2015      1  winter            0    NaN   NaN  22.17   

     SO2     CO     O3  AQI        data_source  temperature_2m_max  \
0   3.54   0.33  10.73  NaN  kaggle_historical                26.5   
1   3.97   1.35  22.77  NaN  kaggle_historical                26.2   
2  13.61  17.40  12.03  NaN  kaggle_historical                26.6   
3   6.52   5.05  17.70  NaN  kaggle_historical                27.1   
4   4.71   1.83  12.72  NaN  kaggle_historical            

In [34]:
hist_mask = master['data_source'] == 'kaggle_historical'
print(f"Historical AQI missing: {master.loc[hist_mask, 'AQI'].isna().sum()} / {hist_mask.sum()} "
      f"({master.loc[hist_mask, 'AQI'].isna().mean()*100:.1f}%)")

Historical AQI missing: 2043 / 12717 (16.1%)


In [35]:
print(master[master['data_source']=='kaggle_historical'].groupby('city')['AQI'].apply(lambda x: x.isna().mean()*100).round(1))

city
Bengaluru     4.9
Chennai       6.2
Delhi         0.5
Kolkata       7.4
Lucknow       5.8
Mumbai       61.4
Patna        21.5
Name: AQI, dtype: float64


In [36]:
import os
os.makedirs('data/processed', exist_ok=True)
master.to_csv('data/processed/aqi_weather_master.csv', index=False)
print(f"Saved {len(master)} rows to data/processed/aqi_weather_master.csv")

Saved 15283 rows to data/processed/aqi_weather_master.csv


In [37]:
readme_content = """# AQI Weather Master Dataset — Data Documentation

## Data Sources
- `data/raw/kaggle/city_day.csv` — CPCB historical air quality data (city-day granularity),
  covering 2015-01-01 to 2020-07-01, 26 Indian cities.
- `data/raw/openaq/openaq_recent.csv` — OpenAQ recent air quality data, long format
  (one row per sensor-parameter-day), covering 2025-08-01 to 2026-08-10, 7 target cities.
- `data/raw/weather/weather_historical.csv` — Open-Meteo weather data matching the Kaggle date range.
- `data/raw/weather/weather_recent.csv` — Open-Meteo weather data matching the OpenAQ date range.

## Target Cities
Bengaluru, Chennai, Delhi, Kolkata, Lucknow, Mumbai, Patna

`weather_historical.csv` originally covered 26 cities nationally; it was filtered down to
the 7 target cities early in the pipeline, since the other 19 have no corresponding
pollutant data in the OpenAQ track.

## City Name Standardization
City names were inconsistent in spelling/format across sources (e.g. "Bangalore" vs
"Bengaluru"). A single mapping dictionary (`city_mapping`) was applied identically to all
four raw dataframes before any merging, to avoid silent row loss from mismatched keys.

## Date Parsing & Derived Features
All date columns were converted to `datetime64`. From each date, the following were
derived and reused throughout: `year`, `month`, `day_of_week` (0=Monday), and `season`
(winter = Nov-Feb, summer = Mar-Jun, monsoon = Jul-Sep, post_monsoon = Oct).

## OpenAQ Reshaping
`openaq_recent.csv` was supplied in long format (one row per sensor-parameter-day).
It was pivoted to wide format (one row per city-day, one column per pollutant) to match
the Kaggle schema, averaging across multiple sensors/readings within the same city-day
(`aggfunc='mean'`). Pollutant parameter names were mapped to Kaggle's naming convention
(e.g. `pm25` -> `PM2.5`, `no2` -> `NO2`) for consistency.

## Historical vs Recent: No Date Overlap
The Kaggle/weather_historical track (2015-2020) and the OpenAQ/weather_recent track
(2025-2026) do not overlap in time at all. They were therefore cleaned and merged as two
independent tracks, each merged against its matching weather file, then concatenated
into one master table with a `data_source` column (`kaggle_historical` /
`openaq_recent`) to preserve provenance. This gap is a known structural feature of the
project's data, not a bug.

## Missing Data Policy
A two-tier rule was applied per city-pollutant pair:
1. **Active-window scoping**: missing % is computed only from a pollutant's first valid
   reading onward for that city, not from the dataset's global start date. This avoids
   penalizing a pollutant for the period before its monitoring station existed. City-
   pollutant pairs with no valid reading at all are dropped outright (100% missing).
2. **Threshold**: pairs with >=40% missing within their active window are excluded from
   analysis (set to NaN). Pairs below the threshold have short gaps (<=2 consecutive days)
   forward-filled per city; longer gaps are left as NaN rather than fabricated.

### Why active-window scoping mattered (concrete example)
Under a naive "missing % from the dataset's start date" rule, Mumbai's PM2.5, PM10, NO,
NO2, SO2, and O3 all showed 60-62% missing and would have been dropped entirely -
removing nearly all core pollutants for one of the 7 target cities. Investigation showed
these stations simply came online between Dec 2017 and Apr 2018; data from that point
onward is ~100% complete. Rescoping missing-% calculation to each pollutant's active
window correctly retained all six columns for Mumbai. Only Mumbai-Toluene (49.4%) and
Mumbai-Xylene (49.5%) remained excluded even within their active window.

### Dropped city-pollutant pairs (Kaggle track, final)
Includes: Ahmedabad-NH3, Bhopal-{Benzene,Toluene,Xylene}, several cities' Xylene columns
(mostly 100% missing - never measured), Jorapokhar-{NOx,CO}, Lucknow-PM10,
Mumbai-{Toluene,Xylene}, Talcher-Toluene, Thiruvananthapuram-{Benzene,Toluene}, and others.
Full list captured in the cleaning notebook's `dropped_kaggle_v2` output.

### Dropped city-pollutant pairs (OpenAQ track, final)
Patna-NO2, Patna-SO2, Patna-CO — all ~87-100% missing throughout their active window
(not a late-start issue; this sensor appears to have been unreliable for its full
operating period, unlike Mumbai's case above).

## Sensor Error Cleanup (OpenAQ track only)
Before AQI computation, raw OpenAQ values were checked against physically plausible
bounds:
- Negative concentrations (impossible): 2 rows each for NO2, CO, O3 (~0.1% of rows).
- Extreme outliers: PM2.5 > 1000 ug/m3 (11 rows, 0.4%), PM10 > 1000 ug/m3 (12 rows, 0.5%).
All flagged values were set to NaN and passed back through the same missing-data rule
above (short-gap forward-fill / threshold-drop). No new city-pollutant pairs crossed the
drop threshold as a result.

## Merge (Step 5)
Two separate inner joins on `(city, date)`:
- `city_day` (filtered to 7 target cities) x `weather_hist` -> 12,717 rows, 0 rows dropped
  (lossless once scoped to target cities; the pre-filter version showed a 57% drop that
  was entirely attributable to the 19 out-of-scope cities, not a real data gap).
- `openaq_wide` x `weather_recent` -> 2,566 rows, 0 rows dropped (lossless).

## AQI Calculation (Step 6)
- **Historical track**: AQI is Kaggle's own precomputed value, carried through the merge
  as-is. It has its own missingness (16.1% of historical rows overall), heavily
  concentrated in Mumbai (61.4% missing) and Patna (21.5% missing) - both explained by
  the same late-station-start pattern documented above. Other cities show <8% missing.
- **Recent track**: AQI was computed from scratch using the CPCB breakpoint formula
  (`aqi_calculator.py` / `compute_aqi`), taking the max of all available pollutant
  sub-indices per row (CPCB methodology - AQI reflects the worst pollutant, not an
  average). Validated against known reference points (15 ug/m3 PM2.5 -> sub-index 25;
  45 ug/m3 -> 74.66; values beyond all breakpoints cap at 500). Result: 0 rows with
  uncomputable AQI across all 2,566 recent rows.
- **Known limitation**: historical and recent AQI were not cross-validated against each
  other using the same formula on the same data, so exact numerical consistency between
  the two tracks' AQI methodology is assumed, not independently verified.
- **Sanity check**: city-level recent-track AQI means ranged from 77 (Bengaluru) to 203
  (Delhi), consistent with Delhi's known status as India's most polluted major city -
  treated as a validation signal that the calculation is behaving correctly.

## Final Master Table
Saved to `data/processed/aqi_weather_master.csv` - 15,283 rows total
(12,717 kaggle_historical + 2,566 openaq_recent).

Columns: `city`, `date`, `year`, `month`, `season`, `day_of_week`, `PM2.5`, `PM10`,
`NO2`, `SO2`, `CO`, `O3`, `AQI`, `data_source`, `temperature_2m_max`,
`temperature_2m_min`, `precipitation_sum`, `windspeed_10m_max`,
`relative_humidity_2m_mean`.

Only the 6 pollutants common to both tracks (PM2.5, PM10, NO2, SO2, CO, O3) are included
in the master table for cross-track comparability. Kaggle-only pollutants (NO, NOx, NH3,
Benzene, Toluene, Xylene) remain available in the intermediate `merged_historical`
dataframe for historical-only analysis if needed later, but were not carried into the
unified master file.

## Known Limitations (summary)
- No date overlap between historical (2015-2020) and recent (2025-2026) tracks - any
  trend analysis across the full period must account for this gap explicitly.
- Historical AQI missingness is uneven across cities (worst: Mumbai, Patna) due to
  staggered real-world station installation dates, not random data loss.
- Historical and recent AQI come from two different computation paths (Kaggle's own vs.
  our CPCB implementation) and have not been cross-validated against each other.
- Patna's NO2/SO2/CO (OpenAQ track) and Mumbai's Toluene/Xylene (Kaggle track) are
  excluded entirely due to persistently high missingness even within their active
  monitoring windows.
"""

with open('data/README.md', 'w') as f:
    f.write(readme_content)

print("README written to data/README.md")
print(f"Length: {len(readme_content)} characters")

README written to data/README.md
Length: 8074 characters
